In [1]:
!nvidia-smi

Sun Oct  4 11:53:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q -U transformers peft datasets accelerate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 116.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 45.7 MB/s eta 0:00:00


In [3]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [5]:
from google.colab import files

uploaded = files.upload()

Saving train.jsonl to train.jsonl


In [6]:
import os

print(os.listdir("/content"))

['.config', 'train.jsonl', 'validation.jsonl', 'sample_data']


In [10]:
from huggingface_hub import login

login()

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

info = api.model_info("google/gemma-3-1b-it")
print(info.id)

In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "google/gemma-3-1b-it"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto"
)

print("Model loaded successfully")

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.00GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Model loaded successfully


In [14]:
!pip install -q -U "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 73.3 MB/s eta 0:00:00


In [15]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 1,490,944 || all params: 1,001,376,896 || trainable%: 0.1489


In [16]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={
        "train": "/content/train.jsonl",
        "validation": "/content/validation.jsonl"
    }
)

print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input', 'output'],
        num_rows: 20
    })
    validation: Dataset({
        features: ['input', 'output'],
        num_rows: 5
    })
})


In [17]:
def format_example(example):
    return {
        "text": f"""Classify the following helpdesk ticket into exactly one category.

Categories:
Billing
Login
Delivery

Ticket:
{example["input"]}

Answer:
{example["output"]}"""
    }

dataset = dataset.map(format_example)

print(dataset["train"][0])

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

{'input': 'I was charged twice for the same order', 'output': 'Billing', 'text': 'Classify the following helpdesk ticket into exactly one category.\n\nCategories:\nBilling\nLogin\nDelivery\n\nTicket:\nI was charged twice for the same order\n\nAnswer:\nBilling'}


In [18]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=256,
    )

tokenized_dataset = dataset.map(
    tokenize,
    batched=True,
    remove_columns=dataset["train"].column_names
)

print(tokenized_dataset)

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 20
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 5
    })
})


In [20]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [21]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,4.441876,4.404408
2,4.444027,4.404408
3,4.604845,4.327363


TrainOutput(global_step=9, training_loss=4.553137726253933, metrics={'train_runtime': 32.9266, 'train_samples_per_second': 1.822, 'train_steps_per_second': 0.273, 'total_flos': 10071172915200.0, 'train_loss': 4.553137726253933, 'epoch': 3.0})

In [22]:
ADAPTER_PATH = "/content/helpdesk-lora-adapter"

model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)

print("LoRA adapter saved successfully!")
print(ADAPTER_PATH)

LoRA adapter saved successfully!
/content/helpdesk-lora-adapter


In [23]:
from google.colab import files

uploaded = files.upload()

Saving test.csv to test.csv


In [24]:
import pandas as pd

test_df = pd.read_csv("/content/test.csv")
print(test_df)

  ticket_id                                             ticket expected_label
0      T016  I changed my password but still cannot access ...          Login
1      T021                     My package has not arrived yet       Delivery
2      T011                     I cannot sign in to my account          Login
3      T002               My invoice shows an incorrect amount        Billing
4      T026  I entered the wrong delivery address and need ...       Delivery


In [26]:
import torch
import pandas as pd

ALLOWED_LABELS = ["Billing", "Login", "Delivery"]


def predict(model, ticket):
    prompt = create_prompt(ticket)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    # Find a valid label anywhere in the generated response
    for label in ALLOWED_LABELS:
        if label.lower() in response.lower():
            return label, response

    return None, response


results = []

print("=" * 70)
print("BASE vs LoRA EVALUATION")
print("=" * 70)

for _, row in test_df.iterrows():

    base_prediction, base_raw = predict(
        base_model,
        row["ticket"]
    )

    lora_prediction, lora_raw = predict(
        lora_model,
        row["ticket"]
    )

    actual = row["expected_label"]

    print(f"\nTicket: {row['ticket_id']}")
    print(f"Actual: {actual}")
    print(f"Base Gemma: {base_prediction or 'INVALID'}")
    print(f"Base raw: {base_raw}")
    print(f"LoRA Gemma: {lora_prediction or 'INVALID'}")
    print(f"LoRA raw: {lora_raw}")

    results.append({
        "ticket_id": row["ticket_id"],
        "ticket": row["ticket"],
        "actual_label": actual,
        "base_prediction": base_prediction or "",
        "base_raw_output": base_raw,
        "lora_prediction": lora_prediction or "",
        "lora_raw_output": lora_raw,
    })


comparison_df = pd.DataFrame(results)

base_accuracy = (
    comparison_df["base_prediction"]
    == comparison_df["actual_label"]
).mean()

lora_accuracy = (
    comparison_df["lora_prediction"]
    == comparison_df["actual_label"]
).mean()

base_invalid = (
    comparison_df["base_prediction"] == ""
).sum()

lora_invalid = (
    comparison_df["lora_prediction"] == ""
).sum()


print("\n" + "=" * 70)
print("FINAL COMPARISON")
print("=" * 70)

print(f"Base Gemma accuracy : {base_accuracy * 100:.2f}%")
print(f"LoRA Gemma accuracy : {lora_accuracy * 100:.2f}%")
print(f"Base invalid labels : {base_invalid}")
print(f"LoRA invalid labels : {lora_invalid}")

comparison_path = "/content/comparison.csv"
comparison_df.to_csv(comparison_path, index=False)

print(f"\nComparison saved to: {comparison_path}")


BASE vs LoRA EVALUATION

Ticket: T016
Actual: Login
Base Gemma: Billing
Base raw: Billing

**Explanation:**

The ticket describes a problem related to accessing an account, which falls under
LoRA Gemma: Billing
LoRA raw: Billing

**Explanation:**

The ticket describes a problem related to accessing an account, which falls under

Ticket: T021
Actual: Delivery
Base Gemma: Billing
Base raw: Billing

**Explanation:**

The ticket describes a problem with a package delivery, which falls under
LoRA Gemma: Billing
LoRA raw: Billing

**Explanation:**

The ticket describes a problem with a package delivery, which falls under

Ticket: T011
Actual: Login
Base Gemma: Login
Base raw: Login

Explanation:
The ticket describes a problem with the login process, which falls under the
LoRA Gemma: Login
LoRA raw: Login

Explanation:
The ticket describes a problem with the login process, which falls under the

Ticket: T002
Actual: Billing
Base Gemma: Billing
Base raw: Billing

Explanation: The ticket descri